# Windows Repository Bridge — optional Colab vision worker

This notebook is a manually started worker for the existing Windows Repository Bridge mailbox. It is not a chat UI and does not use a paid inference API.

Before running:
1. In Colab, select a GPU runtime if one is available.
2. Open the Secrets panel and add a secret named QDQWG_TOKEN. Use a fine-grained GitHub token restricted to hashin9973-ux/qdqwg with Contents read/write permission. Enable notebook access for that secret. Never put the token in a notebook cell or GitHub file.
3. Run the worker cell. The first enabled activation may download roughly 5.03 GB of Q4_K_M weights and 1.16 GB of projector into Google Drive, and compile llama.cpp. Later live-runtime activations reuse the loaded server; later runtimes reuse Drive model files.
4. Use the ChatGPT mailbox tools vision_activate, vision_status, vision_analyze_screenshot, vision_wait and vision_deactivate.

The notebook only processes explicitly queued screenshots. When deactivated, it idles and keeps the Colab process/model alive. It does not shut down the runtime. Colab GPU access and session uptime are not guaranteed.

In [ ]:
# @title Run the optional vision worker
import base64
import hashlib
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
import threading
import time
from datetime import datetime, timezone
from pathlib import Path

import requests
from google.colab import drive, userdata

OWNER = "hashin9973-ux"
REPO = "qdqwg"
BRANCH = "bridge-mailbox"
API_ROOT = f"https://api.github.com/repos/{OWNER}/{REPO}/contents/"
MODEL_REPO = "Qwen/Qwen3-VL-8B-Instruct-GGUF"
MODEL_FILE = "Qwen3VL-8B-Instruct-Q4_K_M.gguf"
PROJECTOR_FILE = "mmproj-Qwen3VL-8B-Instruct-F16.gguf"
MODEL_URL = f"https://huggingface.co/{MODEL_REPO}/resolve/main/{MODEL_FILE}"
PROJECTOR_URL = f"https://huggingface.co/{MODEL_REPO}/resolve/main/{PROJECTOR_FILE}"
SERVER_URL = "http://127.0.0.1:8080"
POLL_SECONDS = 5
HEARTBEAT_SECONDS = 20
LEASE_SECONDS = 20 * 60
MAX_SCREENSHOT_BYTES = 600 * 1024
JOB_PATTERN = re.compile(r"^vision-[a-f0-9]{24}$")
VALID_FOLDER_PATHS = {
    "bridge/vision", "bridge/vision/requests", "bridge/vision/results",
    "bridge/vision/reports", "bridge/vision/claims"
}

try:
    TOKEN = userdata.get("QDQWG_TOKEN")
except Exception as exc:
    TOKEN = None
    print("Missing or inaccessible Colab secret QDQWG_TOKEN.")
    print("Open Colab Secrets, add a repository-scoped GitHub token, enable notebook access, then rerun this cell.")
    raise RuntimeError("Google/Colab authentication or QDQWG_TOKEN setup is required; no token was written to the notebook or mailbox.") from exc

if not TOKEN or not isinstance(TOKEN, str):
    raise RuntimeError("QDQWG_TOKEN is empty. Add the secret in Colab Secrets and rerun the cell.")

HEADERS = {
    "Authorization": "Bearer " + TOKEN,
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}
TOKEN = None  # Drop the Python variable reference after constructing request headers.

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def valid_mailbox_path(relative):
    if relative in VALID_FOLDER_PATHS:
        return "directory"
    if relative in ("bridge/vision/control.json", "bridge/vision/runtime.json"):
        return "json"
    if re.fullmatch(r"bridge/vision/(requests|results|claims)/vision-[a-f0-9]{24}\.json", relative):
        return "json"
    if re.fullmatch(r"bridge/vision/reports/vision-[a-f0-9]{24}\.txt", relative):
        return "text"
    if re.fullmatch(r"bridge/artifacts/vision-[a-f0-9]{24}/screenshot\.jpg", relative):
        return "screenshot"
    raise ValueError("Refusing an out-of-scope mailbox path: " + str(relative))

def gh_request(method, relative, body=None, missing_ok=False):
    kind = valid_mailbox_path(relative)
    method = method.upper()
    if method == "GET":
        response = requests.get(API_ROOT + relative, headers=HEADERS,
                                params={"ref": BRANCH}, timeout=20)
    elif method == "PUT":
        if kind == "directory":
            raise ValueError("Refusing to write a directory path.")
        response = requests.put(API_ROOT + relative, headers=HEADERS,
                                json=body, timeout=30)
    elif method == "DELETE":
        if kind != "screenshot":
            raise ValueError("Deletion is permitted only for the exact temporary screenshot artifact path.")
        response = requests.delete(API_ROOT + relative, headers=HEADERS,
                                   json=body, timeout=30)
    else:
        raise ValueError("Unsupported GitHub operation.")
    if response.status_code == 404 and missing_ok:
        return None
    if response.status_code == 409:
        raise RuntimeError("GitHub content update conflict; a fresh read and retry is required.")
    if not response.ok:
        message = ""
        try:
            message = response.json().get("message", "")
        except Exception:
            pass
        raise RuntimeError(f"GitHub API {method} failed ({response.status_code}): {message[:400]}")
    return response.json() if response.content else {}

def read_json(relative, missing_ok=False):
    entry = gh_request("GET", relative, missing_ok=missing_ok)
    if entry is None:
        return None
    if not isinstance(entry, dict) or entry.get("encoding") != "base64" or not isinstance(entry.get("content"), str):
        raise RuntimeError("Unexpected GitHub Contents API response for " + relative)
    decoded = base64.b64decode(entry["content"]).decode("utf-8")
    value = json.loads(decoded)
    return {"value": value, "sha": entry.get("sha")}

def put_json(relative, value, create_only=False):
    raw = json.dumps(value, ensure_ascii=False, indent=2).encode("utf-8")
    for attempt in range(3):
        existing = read_json(relative, missing_ok=True)
        if create_only and existing:
            if existing["value"] == value:
                return {"ok": True, "unchanged": True}
            return {"ok": False, "conflict": True}
        body = {
            "message": "[Colab vision worker] " + relative,
            "content": base64.b64encode(raw).decode("ascii"),
            "branch": BRANCH
        }
        if existing and existing.get("sha"):
            body["sha"] = existing["sha"]
        try:
            gh_request("PUT", relative, body=body)
            return {"ok": True, "unchanged": False}
        except Exception as exc:
            if attempt == 2 or "conflict" not in str(exc).lower():
                raise
            time.sleep(0.4 * (attempt + 1))
    raise RuntimeError("Could not publish JSON record " + relative)

def put_text(relative, text):
    if valid_mailbox_path(relative) != "text":
        raise ValueError("Text reports may only be written to the vision reports folder.")
    raw = text.encode("utf-8")
    for attempt in range(3):
        existing = gh_request("GET", relative, missing_ok=True)
        if existing and existing.get("encoding") == "base64":
            try:
                if base64.b64decode(existing.get("content", "")) == raw:
                    return
            except Exception:
                pass
        body = {
            "message": "[Colab vision worker] publish human-readable report",
            "content": base64.b64encode(raw).decode("ascii"),
            "branch": BRANCH
        }
        if existing and existing.get("sha"):
            body["sha"] = existing["sha"]
        try:
            gh_request("PUT", relative, body=body)
            return
        except Exception as exc:
            if attempt == 2 or "conflict" not in str(exc).lower():
                raise
            time.sleep(0.4 * (attempt + 1))
    raise RuntimeError("Could not publish text report " + relative)

RUNTIME_PATH = "bridge/vision/runtime.json"
_server = None
_model_loaded = False
_gpu = None
_vram_mib = None
_active_request = None
_last_completed = None
_runtime_lock = threading.Lock()
_last_runtime_payload = None
_last_runtime_publish = 0.0
_last_setup_attempt = 0.0

def publish_runtime(status, stage=None, message=None, **extra):
    global _last_runtime_payload, _last_runtime_publish
    with _runtime_lock:
        now = time.monotonic()
        payload = {
            "protocol": "chatgpt-colab-vision-runtime/v1",
            "status": status,
            "heartbeat_at": utc_now(),
            "updated_at": utc_now(),
            "worker": "colab-vision-worker/v1",
            "model": MODEL_FILE,
            "model_repository": MODEL_REPO,
            "projector": PROJECTOR_FILE,
            "model_loaded": bool(_model_loaded),
            "gpu": _gpu,
            "vram_mib": _vram_mib,
            "active_request_id": _active_request,
            "last_completed_request_id": _last_completed,
            "stage": stage,
            "message": (message or "")[:1000]
        }
        payload.update(extra)
        # Heartbeats can be sent every interval, but never publish a success state
        # based only on a browser page or a submitted cell.
        put_json(RUNTIME_PATH, payload)
        _last_runtime_payload = payload
        _last_runtime_publish = now
        return payload

def read_control():
    item = read_json("bridge/vision/control.json", missing_ok=True)
    if not item:
        return {"enabled": False}
    value = item["value"]
    if value.get("protocol") != "chatgpt-colab-vision-control/v1":
        raise RuntimeError("Unsupported vision control protocol.")
    return value

def check_gpu():
    if not shutil.which("nvidia-smi"):
        return None
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader,nounits"],
        check=True, capture_output=True, text=True, timeout=15
    )
    line = next((line.strip() for line in result.stdout.splitlines() if line.strip()), "")
    if not line:
        return None
    columns = [column.strip() for column in line.split(",")]
    if len(columns) < 3:
        raise RuntimeError("Could not parse nvidia-smi GPU/VRAM output.")
    return columns[0], int(columns[1]), int(columns[2])

def ensure_server():
    global _server, _model_loaded, _gpu, _vram_mib, _last_setup_attempt
    if _server is not None and _server.poll() is None:
        try:
            healthy = requests.get(SERVER_URL + "/health", timeout=2)
            if healthy.ok:
                _model_loaded = True
                return True
        except requests.RequestException:
            pass
    if time.monotonic() - _last_setup_attempt < 45:
        return False
    _last_setup_attempt = time.monotonic()
    publish_runtime("starting", stage="drive_mount", message="Preparing persistent model storage.")
    try:
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        publish_runtime("needs_authentication", stage="drive_mount",
                        message="Google Drive authorization is required in the Colab browser: " + str(exc))
        return False

    gpu = check_gpu()
    if not gpu:
        publish_runtime("gpu_unavailable", stage="detect_gpu",
                        message="No NVIDIA GPU is attached to this Colab runtime. Select a GPU runtime if available, then reconnect and rerun the cell.")
        return False
    _gpu, total_mib, free_mib = gpu
    _vram_mib = total_mib
    if total_mib < 10000:
        publish_runtime("gpu_unavailable", stage="check_vram",
                        message=f"Detected {_gpu} with {total_mib} MiB VRAM. The requested 8B model plus F16 projector needs more available memory than this runtime may provide.")
        return False
    if not shutil.which("nvcc"):
        publish_runtime("error", stage="check_cuda",
                        message="An NVIDIA GPU is present but nvcc is missing. The CUDA llama.cpp build cannot be verified in this runtime.")
        return False

    root = Path("/content/drive/MyDrive/WindowsRepositoryBridgeVision")
    model_dir = root / "models"
    hf_cache = root / "hf-cache"
    model_dir.mkdir(parents=True, exist_ok=True)
    hf_cache.mkdir(parents=True, exist_ok=True)
    needed = [
        model_dir / MODEL_FILE,
        model_dir / PROJECTOR_FILE
    ]
    free_bytes = shutil.disk_usage(root).free
    if any(not item.exists() for item in needed) and free_bytes < 7_000_000_000:
        publish_runtime("error", stage="check_drive_space",
                        message=f"Persistent Drive storage has only {free_bytes // (1024**3)} GiB free; at least 7 GB is required for the model and projector.")
        return False

    publish_runtime("starting", stage="dependencies", message="Checking one-time model and build dependencies.")
    if importlib.util.find_spec("huggingface_hub") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True, timeout=300)
    from huggingface_hub import hf_hub_download

    publish_runtime("starting", stage="download_model", message="Reusing or downloading the pre-quantized Q4_K_M model and matching F16 vision projector.")
    model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE,
                                 local_dir=str(model_dir), cache_dir=str(hf_cache))
    projector_path = hf_hub_download(repo_id=MODEL_REPO, filename=PROJECTOR_FILE,
                                     local_dir=str(model_dir), cache_dir=str(hf_cache))
    if Path(model_path).stat().st_size < 4_500_000_000 or Path(projector_path).stat().st_size < 1_000_000_000:
        raise RuntimeError("Model or projector file is unexpectedly small; refusing to start inference with a partial download.")

    llama_source = Path("/content/llama.cpp")
    if not (llama_source / "CMakeLists.txt").exists():
        publish_runtime("starting", stage="clone_llama_cpp", message="Fetching the current upstream llama.cpp source.")
        subprocess.run(["git", "clone", "--depth=1", "https://github.com/ggml-org/llama.cpp.git", str(llama_source)],
                       check=True, timeout=600)
    build_dir = llama_source / "build"
    server_binary = build_dir / "bin" / "llama-server"
    if not server_binary.exists():
        if not shutil.which("cmake") or not shutil.which("make"):
            subprocess.run(["apt-get", "update", "-qq"], check=True, timeout=300)
            subprocess.run(["apt-get", "install", "-y", "-qq", "build-essential", "cmake", "git"],
                           check=True, timeout=600)
        publish_runtime("starting", stage="build_llama_cpp", message="Building llama-server with CUDA and multimodal support.")
        subprocess.run(["cmake", "-S", str(llama_source), "-B", str(build_dir),
                        "-DGGML_CUDA=ON", "-DCMAKE_BUILD_TYPE=Release", "-DGGML_NATIVE=OFF"],
                       check=True, timeout=900)
        subprocess.run(["cmake", "--build", str(build_dir), "--config", "Release",
                        "--target", "llama-server", "-j", "2"],
                       check=True, timeout=1800)
    if not server_binary.exists():
        raise RuntimeError("The llama.cpp build ended without a llama-server executable.")
    help_text = subprocess.run([str(server_binary), "--help"], check=True, capture_output=True,
                               text=True, timeout=30).stdout
    for expected_flag in ("--mmproj", "--host", "--port", "--ctx-size", "--n-gpu-layers"):
        if expected_flag not in help_text:
            raise RuntimeError("The built llama-server does not expose the required CLI flag: " + expected_flag)

    if _server is not None and _server.poll() is None:
        _server.terminate()
        try:
            _server.wait(timeout=10)
        except subprocess.TimeoutExpired:
            _server.kill()
    log_path = Path("/content/llama-server-vision.log")
    log_handle = open(log_path, "a", encoding="utf-8")
    _server = subprocess.Popen([
        str(server_binary), "-m", str(model_path), "--mmproj", str(projector_path),
        "-c", "8192", "--n-gpu-layers", "99", "--host", "127.0.0.1", "--port", "8080"
    ], stdout=log_handle, stderr=subprocess.STDOUT, start_new_session=True)
    publish_runtime("starting", stage="load_model", message="llama-server started; waiting for /health to confirm model loading.")
    deadline = time.monotonic() + 240
    while time.monotonic() < deadline:
        if _server.poll() is not None:
            try:
                tail = log_path.read_text(encoding="utf-8", errors="replace")[-2500:]
            except Exception:
                tail = "(server log unavailable)"
            raise RuntimeError("llama-server exited before reporting healthy. Diagnostic tail: " + tail)
        try:
            health = requests.get(SERVER_URL + "/health", timeout=3)
            if health.ok:
                _model_loaded = True
                _vram_mib = total_mib
                publish_runtime("ready", stage="ready", message="The model and vision projector are loaded; llama-server health check succeeded.")
                log_handle.close()
                return True
        except requests.RequestException:
            pass
        time.sleep(2)
    raise RuntimeError("Timed out waiting for llama-server /health after loading the Qwen3-VL model.")

def list_jobs():
    entry = gh_request("GET", "bridge/vision/requests", missing_ok=True)
    return entry if isinstance(entry, list) else []

def download_screenshot(relative, expected_hash):
    if valid_mailbox_path(relative) != "screenshot":
        raise ValueError("Request referred to a screenshot outside the dedicated vision artifact directory.")
    entry = gh_request("GET", relative)
    if entry.get("encoding") != "base64" or not isinstance(entry.get("content"), str):
        raise RuntimeError("Screenshot artifact returned an unsupported encoding.")
    data = base64.b64decode(entry["content"])
    if not data or len(data) > MAX_SCREENSHOT_BYTES:
        raise RuntimeError("Screenshot artifact is empty or exceeds the 600 KiB inference limit.")
    digest = hashlib.sha256(data).hexdigest()
    if digest != expected_hash:
        raise RuntimeError("Screenshot SHA-256 did not match the queued request; inference was refused.")
    return data, entry.get("sha")

def cleanup_screenshot(relative):
    # Restrict cleanup to the per-job temporary JPEG and use its current blob SHA.
    if not isinstance(relative, str) or valid_mailbox_path(relative) != "screenshot":
        return False
    try:
        entry = gh_request("GET", relative, missing_ok=True)
        if not entry or not entry.get("sha"):
            return True
        gh_request("DELETE", relative, body={
            "message": "[Colab vision worker] remove temporary screenshot after processing",
            "sha": entry["sha"], "branch": BRANCH
        })
        return True
    except Exception as exc:
        print("Temporary screenshot cleanup did not complete:", str(exc)[:300])
        return False

def make_result(request, report, status, started_at, error=None):
    job_id = request["request_id"]
    report_path = f"bridge/vision/reports/{job_id}.txt"
    completed = utc_now()
    result = {
        "protocol": "chatgpt-colab-vision-result/v1",
        "request_id": job_id,
        "status": status,
        "created_at": request.get("created_at"),
        "completed_at": completed,
        "model_repository": MODEL_REPO,
        "model": MODEL_FILE,
        "input_sha256": request.get("sha256"),
        "input_image": {
            "sha256": request.get("sha256"),
            "content_type": request.get("content_type", "image/jpeg"),
            "bytes": request.get("bytes")
        },
        "inference_ms": int((time.monotonic() - started_at) * 1000)
    }
    if status == "completed":
        result["report"] = report
        result["report_path"] = report_path
    else:
        result["error"] = (error or "Vision inference failed.")[:1800]
    return result

def run_one_request(request):
    global _active_request, _last_completed
    job_id = request.get("request_id")
    if not isinstance(job_id, str) or not JOB_PATTERN.fullmatch(job_id):
        raise ValueError("Queued request ID is malformed.")
    request_path = f"bridge/vision/requests/{job_id}.json"
    current = read_json(request_path)
    if not current or current["value"].get("protocol") != "chatgpt-colab-vision-request/v1":
        raise RuntimeError("Vision request protocol did not match.")
    request = current["value"]
    if request.get("request_id") != job_id or request.get("idempotency_key") != job_id:
        raise RuntimeError("Vision request ID or idempotency key did not match its filename.")
    existing_result = read_json(f"bridge/vision/results/{job_id}.json", missing_ok=True)
    image_path = request.get("image_path")
    if existing_result:
        cleanup_screenshot(image_path)
        return

    claim_path = f"bridge/vision/claims/{job_id}.json"
    prior_claim = read_json(claim_path, missing_ok=True)
    current_seconds = time.time()
    if prior_claim:
        claim = prior_claim["value"]
        if claim.get("status") == "completed":
            return
        lease = float(claim.get("lease_expires_at", 0) or 0)
        if claim.get("status") == "processing" and lease > current_seconds:
            return
    claim = {
        "protocol": "chatgpt-colab-vision-claim/v1",
        "request_id": job_id,
        "status": "processing",
        "claimed_at": utc_now(),
        "lease_expires_at": current_seconds + LEASE_SECONDS,
        "worker": "colab-vision-worker/v1"
    }
    put_json(claim_path, claim, create_only=False)
    _active_request = job_id
    publish_runtime("inference_in_progress", stage="download_screenshot",
                    message="Fetching one queued screenshot and verifying its SHA-256.")
    start = time.monotonic()
    result = None
    try:
        if not _model_loaded:
            raise RuntimeError("Model readiness was lost before inference; restart the setup cell and retry this same request ID.")
        image_bytes, _image_blob_sha = download_screenshot(image_path, request.get("sha256"))
        prompt = str(request.get("prompt") or "Describe the visible screenshot evidence. Include overview, controls/layout, exact legible text, visible errors/anomalies, evidence, and uncertainty.")[:2000]
        image_data = base64.b64encode(image_bytes).decode("ascii")
        publish_runtime("inference_in_progress", stage="inference",
                        message="The loaded Qwen3-VL model is analyzing the requested screenshot.")
        response = requests.post(
            SERVER_URL + "/v1/chat/completions",
            json={
                "model": "qwen3-vl-8b-vision",
                "temperature": 0.1,
                "max_tokens": 1800,
                "stream": False,
                "messages": [
                    {"role": "system", "content": (
                        "You are a careful visual observer, not a general reasoning agent. "
                        "Describe only details supported by the provided image. Do not infer unseen state. "
                        "Mark illegible text and uncertainty explicitly. Organize the report into: Overview; "
                        "Visible objects, panels, controls and layout; Text transcription; Visible errors or anomalies; "
                        "Evidence supporting observations; Uncertainties."
                    )},
                    {"role": "user", "content": [
                        {"type": "text", "text": prompt},
                        {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + image_data}}
                    ]}
                ]
            }, timeout=(10, 300)
        )
        if not response.ok:
            raise RuntimeError(f"llama-server inference endpoint returned HTTP {response.status_code}: {response.text[:800]}")
        payload = response.json()
        choices = payload.get("choices")
        if not isinstance(choices, list) or not choices or not isinstance(choices[0], dict):
            raise RuntimeError("llama-server returned a malformed completion payload.")
        message = choices[0].get("message", {})
        content = message.get("content") if isinstance(message, dict) else None
        if isinstance(content, list):
            content = "\n".join(str(part.get("text", "")) for part in content if isinstance(part, dict))
        if not isinstance(content, str) or not content.strip():
            raise RuntimeError("The model returned an empty or non-text report.")
        report = content.strip()
        if len(report) > 30000:
            report = report[:30000] + "\n\n[Report truncated at the safety limit.]"
        # Publish the readable report first. The structured success result is the
        # commit marker and is written only after report publication succeeds.
        put_text(f"bridge/vision/reports/{job_id}.txt", report + "\n")
        result = make_result(request, report, "completed", start)
    except Exception as exc:
        result = make_result(request, None, "failed", start,
                             error=type(exc).__name__ + ": " + str(exc))
    try:
        put_json(f"bridge/vision/results/{job_id}.json", result, create_only=True)
    except Exception as exc:
        # If a previous publish committed but its HTTP response was lost, verify
        # the matching immutable record before considering this request delivered.
        recovered = read_json(f"bridge/vision/results/{job_id}.json", missing_ok=True)
        if not recovered or recovered["value"].get("request_id") != job_id or recovered["value"].get("input_sha256") != request.get("sha256"):
            raise RuntimeError("Could not publish or recover the result record: " + str(exc)) from exc
        result = recovered["value"]
    put_json(claim_path, {
        "protocol": "chatgpt-colab-vision-claim/v1",
        "request_id": job_id,
        "status": "completed",
        "completed_at": utc_now(),
        "result_status": result.get("status"),
        "input_sha256": request.get("sha256"),
        "worker": "colab-vision-worker/v1"
    }, create_only=False)
    cleanup_screenshot(image_path)
    _last_completed = job_id
    _active_request = None
    publish_runtime("ready", stage="ready", message="Inference result was published and checked against the request ID and input SHA-256.")
    print("Vision request completed:", job_id, "| status:", result.get("status"), "| inference ms:", result.get("inference_ms"))

def reconcile_pending():
    # Process jobs committed before a Colab disconnect. Results and claim leases
    # are durable mailbox records, so a new runtime can resume without resubmitting.
    entries = list_jobs()
    entries = [item for item in entries if isinstance(item, dict) and isinstance(item.get("name"), str)
               and item["name"].endswith(".json") and JOB_PATTERN.fullmatch(item["name"][:-5])]
    entries.sort(key=lambda item: item["name"])
    for item in entries:
        job_id = item["name"][:-5]
        request_path = f"bridge/vision/requests/{job_id}.json"
        record = read_json(request_path, missing_ok=True)
        if not record:
            continue
        request = record["value"]
        result = read_json(f"bridge/vision/results/{job_id}.json", missing_ok=True)
        if result:
            cleanup_screenshot(request.get("image_path", ""))
            continue
        created = request.get("created_at")
        try:
            age = time.time() - datetime.fromisoformat(created.replace("Z", "+00:00")).timestamp()
        except Exception:
            age = 0
        if age > 24 * 3600:
            failure = make_result(request, None, "failed", time.monotonic(),
                                  error="Request expired after 24 hours without a completed result. Re-activate vision and submit a new screenshot if still needed.")
            put_json(f"bridge/vision/results/{job_id}.json", failure, create_only=True)
            cleanup_screenshot(request.get("image_path", ""))
            continue
        run_one_request(request)

def run_worker():
    print("Colab vision worker started. It will not load the model until vision_activate sets enabled=true.")
    last_heartbeat = 0.0
    last_reconcile = 0.0
    while True:
        try:
            control = read_control()
            enabled = control.get("enabled") is True
            now = time.monotonic()
            if enabled and not _model_loaded:
                if not ensure_server():
                    time.sleep(POLL_SECONDS)
                    continue
            if not enabled:
                status = "inactive"
                message = "Vision is disabled; no screenshots will be processed. The Colab runtime is intentionally kept alive."
                if now - last_heartbeat >= HEARTBEAT_SECONDS:
                    publish_runtime(status, stage="idle", message=message)
                    last_heartbeat = now
                time.sleep(POLL_SECONDS)
                continue
            if not _model_loaded:
                time.sleep(POLL_SECONDS)
                continue
            if now - last_reconcile >= POLL_SECONDS:
                publish_runtime("ready", stage="polling", message="Vision enabled; checking the existing mailbox for explicitly queued screenshots.")
                reconcile_pending()
                last_reconcile = time.monotonic()
            if time.monotonic() - last_heartbeat >= HEARTBEAT_SECONDS:
                publish_runtime("ready", stage="polling", message="Vision is ready and polling the mailbox.")
                last_heartbeat = time.monotonic()
        except KeyboardInterrupt:
            print("Notebook cell interrupted. The Colab runtime is not being shut down.")
            break
        except Exception as exc:
            message = type(exc).__name__ + ": " + str(exc)
            print("Vision worker recoverable error:", message[:800])
            try:
                publish_runtime("error", stage="worker_error", message=message)
            except Exception as publish_exc:
                print("Could not publish error state:", str(publish_exc)[:300])
            time.sleep(POLL_SECONDS)

run_worker()
